# Travel Planner Prompt 优化教程（textgrad 计算图 + L1/L2 评测）

> 内部代号也叫 **Planner B1**（与 **Planner B2 / E2E 优化** 区分）。本 Notebook 面向**第一次接触本仓库优化流程**的读者，按 Step 1→4 动手跑通一遍。

## 你将学到什么

1. 什么叫 **L1 / L2** 评测，优化前后如何对比分数
2. 如何用 **`textgrad_graph`** 优化 Planner 的两段 prompt（`decomposition_prompt`、`agent_routing`）
3. 优化产物写到哪里、报告里 **接受 / 拒绝** 表示什么
4. 与「完整 E2E 优化」（B2）的区别——本教程**不跑**整条 LangGraph 编排

## 核心概念（先读这一段）

| 术语 | 含义 |
|------|------|
| **L1** | 任务拆解评测：`decomposition_prompt` 拆出的子任务是否合理 |
| **L2** | 子任务路由评测：`agent_routing` 是否把子任务分给对的 Agent |
| **textgrad_graph** | 把 TaskPlanner 三步（拆解→依赖→路由）包成计算图，用 TextGrad 反传改 prompt |
| **slot** | 两个可优化位置：`decomposition`（L1）、`routing`（L2） |
| **rollback** | 候选 prompt 在 dev 上没变好就拒绝，保留上一轮最优 |

**生产路径不动**：demo / LangGraph 仍走原流程；优化结果写入 `data/benchmark/travel_planner/optimized/zh.json`，运行时自动加载。

## 数据流示意

```mermaid
flowchart LR
  A[benchmark fixtures] --> B[Step1 基线 L1/L2]
  B --> C[Step2 textgrad_graph 优化]
  C --> D[optimized/zh.json + report.json]
  D --> E[Step4 复评 L1/L2]
  E --> F[与 Step1 对比]
```

## 四步流程

| Step | 内容 | 学习者应看到什么 |
|------|------|------------------|
| **Step 1** | L1/L2 **基线**评测（优化前） | 每条 case 的 PASS/FAIL 与均分 |
| **Step 2** | `textgrad_graph` **优化** | 每 slot 的 baseline_dev / best_dev；可能「无失败 case」提前结束 |
| **Step 3** | 查看优化**报告** | 逐步接受/拒绝记录 + prompt 预览 |
| **Step 4** | 优化后 **复评** | L1/L2 分数相对 Step 1 的变化 |

## 前置条件

```powershell
cd Chapter-11
pip install -e ".[evolution]"
pip install nest_asyncio   # Jupyter 下建议安装
```

`.env` 中配置 `DASHSCOPE_API_KEY`（或 `OPENAI_API_KEY`）。

## 默认参数说明（smoke）

下面默认 `TRAIN_SPLIT=dev`、`DEV_SPLIT=dev`、`MAX_STEPS=1`：**只用 2 条 dev case、每 slot 最多 1 轮**，省 API、便于试跑。

- 正式学习/实验请改为：`TRAIN_SPLIT="train"`、`DEV_SPLIT="dev"`、`MAX_STEPS=3`
- 若 Step 2 很快结束且 `failure_count=0`，表示当前 prompt 在 train 上已无低分 case，**不会强行改 prompt**（属正常）
- Step 1 若 L1/L2 已接近满分，Step 4 对比可能几乎无变化——换 `train` split 或加大 `MAX_STEPS` 再试

In [6]:
# 环境与公共配置
import asyncio
import json
import os
import sys
from pathlib import Path

# Notebook 在 notebooks/ 下，项目根为上一级
ROOT = Path.cwd().resolve()
if not (ROOT / "agent_framework").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from agent_framework.config import create_llm, load_project_dotenv
from agent_framework.optimization.core.save import save_planner_optimization_artifacts
from agent_framework.optimization.decomposition.evaluator import evaluate_decomposition_benchmark
from agent_framework.optimization.decomposition.fixtures import load_decomposition_fixtures
from agent_framework.optimization.planner_pipeline import parse_planner_slots, run_planner_optimization
from agent_framework.optimization.planner_runtime import build_planner
from agent_framework.optimization.prompt_store import load_optimized_prompt_payload, optimized_prompts_path
from agent_framework.optimization.routing.evaluator import evaluate_routing_benchmark
from domains.travel.prompt_bundle import TravelPrompts
from domains.travel.specs import create_travel_registry_stub

load_project_dotenv()

# Jupyter 里允许多次 asyncio.run / await
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

OUTPUT_DIR = ROOT / "data" / "benchmark" / "travel_planner"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OPTIMIZED_FILE = optimized_prompts_path("zh")
REPORT_FILE = OUTPUT_DIR / "planner_textgrad_graph_optimization_report.json"
BASELINE_FILE = OUTPUT_DIR / "baseline_l1_l2_dev.json"

FIXTURES = load_decomposition_fixtures()
REGISTRY = create_travel_registry_stub()
BASE_PROMPTS = TravelPrompts.build(locale=FIXTURES.locale, use_optimized=False)

EXECUTOR_MODEL = os.getenv("EXECUTOR_MODEL") or os.getenv("DASHSCOPE_CHAT_MODEL") or "qwen-plus"
OPTIMIZER_MODEL = os.getenv("OPTIMIZER_MODEL") or EXECUTOR_MODEL
EXECUTOR_LLM = create_llm(temperature=0, model=EXECUTOR_MODEL)
OPTIMIZER_LLM = create_llm(temperature=0.2, model=OPTIMIZER_MODEL)

# smoke 参数（可改）
TRAIN_SPLIT = "dev"
DEV_SPLIT = "dev"
MAX_STEPS = 1
FAILURE_THRESHOLD = 0.8

print(f"项目根: {ROOT}")
print(f"fixtures: {len(FIXTURES.cases)} cases, train={len(FIXTURES.splits.get('train', []))}, dev={len(FIXTURES.splits.get('dev', []))}")
print(f"executor={EXECUTOR_MODEL} optimizer={OPTIMIZER_MODEL}")
print(f"optimized 输出: {OPTIMIZED_FILE}")

项目根: D:\myproject\mira-ai-lab\agent-systems-book\Chapter-11
fixtures: 15 cases, train=7, dev=2
executor=qwen-plus optimizer=qwen-plus
optimized 输出: D:\myproject\mira-ai-lab\agent-systems-book\Chapter-11\data\benchmark\travel_planner\optimized\zh.json


## Step 1　基线评测（优化前 L1/L2）

在默认 prompt 上跑 benchmark，得到优化前的 **L1（decomposition）** 与 **L2（routing）** 分数；本步不调用 textgrad。

In [7]:
async def run_baseline_l1_l2(split: str = "dev", use_optimized: bool = False):
    """跑 L1 decomposition + L2 routing benchmark。"""
    overrides = {}
    if use_optimized:
        payload = load_optimized_prompt_payload(FIXTURES.locale)
        overrides = {
            k: str(payload[k])
            for k in ("decomposition_prompt", "agent_routing")
            if str(payload.get(k) or "").strip()
        }

    planner = build_planner(
        EXECUTOR_LLM,
        REGISTRY,
        locale=FIXTURES.locale,
        prompt_overrides=overrides or None,
        use_optimized=use_optimized and not overrides,
    )
    l1 = await evaluate_decomposition_benchmark(
        planner, registry=REGISTRY, fixtures=FIXTURES, split=split
    )
    l2 = await evaluate_routing_benchmark(planner, fixtures=FIXTURES, split=split)
    return {"l1": l1, "l2": l2}


def print_l1_l2_report(tag: str, result: dict) -> None:
    l1, l2 = result["l1"], result["l2"]
    print(f"\n=== {tag} ===")
    print(f"L1 decomposition: avg={l1.average_score:.3f} cases={l1.case_count}")
    for item in l1.cases:
        status = "PASS" if item.score.total >= 0.8 else "FAIL"
        print(f"  [{status}] {item.case_id} score={item.score.total:.3f} subtasks={item.score.subtask_count}")
    print(f"L2 routing:       avg={l2.average_score:.3f} cases={l2.case_count}")
    for item in l2.cases:
        status = "PASS" if item.score.total >= 0.8 else "FAIL"
        agents = ", ".join(f"{t.get('task_id')}->{t.get('agent')}" for t in item.subtasks[:4])
        print(f"  [{status}] {item.case_id} score={item.score.total:.3f} {agents}")


step1 = await run_baseline_l1_l2(split=DEV_SPLIT, use_optimized=False)
print_l1_l2_report("Step1 基线（优化前）", step1)

BASELINE_FILE.write_text(
    json.dumps({"l1": step1["l1"].to_dict(), "l2": step1["l2"].to_dict()}, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print(f"\n已保存基线报告: {BASELINE_FILE}")


=== Step1 基线（优化前） ===
L1 decomposition: avg=0.950 cases=2
  [PASS] case-08 score=0.900 subtasks=3
  [PASS] case-09 score=1.000 subtasks=2
L2 routing:       avg=1.000 cases=2
  [PASS] case-08 score=1.000 T1->WeatherAgent, T2->HotelAgent, T3->RestaurantAgent
  [PASS] case-09 score=1.000 T1->FlightAgent, T2->HotelAgent

已保存基线报告: D:\myproject\mira-ai-lab\agent-systems-book\Chapter-11\data\benchmark\travel_planner\baseline_l1_l2_dev.json


## Step 2　运行 textgrad_graph 优化（L1/L2 目标）

在 **train split** 上找得分低于阈值的 case → 对失败 case 做 graph backward → 用 **dev split** 分数决定是否接受新 prompt（rollback）。

- `backend="textgrad_graph"`：计算图优化（本教程重点）
- `objective="l1_l2"`：只看 L1/L2 分，**不**跑完整 E2E（那是 B2 / `--objective e2e`）
- `slots=all`：顺序优化 **decomposition → routing**

需要已安装 `textgrad` 与 LLM API；smoke 配置大约数分钟。

In [8]:
async def run_planner_graph_optimization():
    """textgrad_graph + l1_l2：优化 decomposition 与 routing 两段 prompt。"""
    return await run_planner_optimization(
        backend="textgrad_graph",
        slots=parse_planner_slots("all"),
        decomposition_prompt=BASE_PROMPTS.decomposition_prompt,
        agent_routing=BASE_PROMPTS.agent_routing,
        registry=REGISTRY,
        executor_llm=EXECUTOR_LLM,
        optimizer_llm=OPTIMIZER_LLM,
        fixtures=FIXTURES,
        max_steps=MAX_STEPS,
        failure_threshold=FAILURE_THRESHOLD,
        rollback=True,
        train_split=TRAIN_SPLIT,
        dev_split=DEV_SPLIT,
        objective="l1_l2",
    )


print(
    f"开始优化: backend=textgrad_graph objective=l1_l2 "
    f"train={TRAIN_SPLIT} dev={DEV_SPLIT} max_steps={MAX_STEPS} "
    f"failure_threshold={FAILURE_THRESHOLD}"
)
step2 = await run_planner_graph_optimization()

save_planner_optimization_artifacts(
    locale=FIXTURES.locale,
    output_path=OPTIMIZED_FILE,
    report_path=REPORT_FILE,
    executor_model=EXECUTOR_MODEL,
    optimizer_model=OPTIMIZER_MODEL,
    backend="textgrad_graph",
    decomposition_result=step2.decomposition_result,
    routing_result=step2.routing_result,
)

for label, result in (
    ("decomposition (L1)", step2.decomposition_result),
    ("routing (L2)", step2.routing_result),
):
    if result is None:
        continue
    print(
        f"{label}: baseline_dev={result.baseline_dev_score:.3f} "
        f"best_dev={result.best_dev_score:.3f} optimizer={result.optimizer}"
    )
    if result.steps and result.steps[-1].failure_count == 0:
        print("  → 无失败 case，本 slot 提前结束（prompt 未改动）")

print(f"\n已写入 prompt: {OPTIMIZED_FILE}")
print(f"已写入报告:   {REPORT_FILE}")

开始优化: backend=textgrad_graph objective=l1_l2 train=dev dev=dev max_steps=1 failure_threshold=0.8
decomposition (L1): baseline_dev=0.950 best_dev=0.950 optimizer=textgrad_graph
  → 无失败 case，本 slot 提前结束（prompt 未改动）
routing (L2): baseline_dev=1.000 best_dev=1.000 optimizer=textgrad_graph
  → 无失败 case，本 slot 提前结束（prompt 未改动）

已写入 prompt: D:\myproject\mira-ai-lab\agent-systems-book\Chapter-11\data\benchmark\travel_planner\optimized\zh.json
已写入报告:   D:\myproject\mira-ai-lab\agent-systems-book\Chapter-11\data\benchmark\travel_planner\planner_textgrad_graph_optimization_report.json


## Step 3　查看优化结果

阅读 JSON 报告中的逐步**接受 / 拒绝**，以及 `optimized/zh.json` 里的 prompt 预览。

> 若 Step 2 未实际改 prompt，报告里可能只有 baseline 分数、没有 step 记录——请先确认 Step 2 是否 `failure_count=0` 提前结束。

In [10]:
def print_optimization_summary(report_path: Path, optimized_path: Path) -> dict:
    """阅读优化报告：逐步得分、是否接受候选 prompt、optimized 文本预览。"""
    report = json.loads(report_path.read_text(encoding="utf-8")) if report_path.is_file() else {}
    opt = json.loads(optimized_path.read_text(encoding="utf-8")) if optimized_path.is_file() else {}

    slot_labels = {
        "decomposition": "decomposition (L1 任务拆解)",
        "routing": "routing (L2 子任务路由)",
    }

    print("=" * 60)
    print("Planner Prompt 优化报告摘要")
    print("=" * 60)
    print(f"backend: {report.get('backend')}")
    print(f"executor_model: {report.get('executor_model')}")
    print(f"optimizer_model: {report.get('optimizer_model')}")

    for slot in ("decomposition", "routing"):
        block = report.get(slot)
        if not block:
            continue
        print(f"\n--- {slot_labels.get(slot, slot)} ---")
        print(f"  optimizer: {block.get('optimizer')}")
        print(f"  baseline_dev: {block.get('baseline_dev_score')}  （优化前 dev 均分）")
        print(f"  best_dev:     {block.get('best_dev_score')}  （最终保留的 dev 均分）")
        for s in block.get("steps") or []:
            flag = "接受" if s.get("accepted") else "拒绝"
            print(
                f"  step{s['step']} [{flag}] train={s['train_average']:.3f} "
                f"candidate_dev={s['candidate_dev_average']:.3f} failures={s['failure_count']}"
            )
            if s.get("failure_count") == 0:
                print("    （无失败 case，本步未更新 prompt）")

    for key in ("decomposition_prompt", "agent_routing"):
        text = str(opt.get(key) or "")
        if text:
            print(f"\n--- optimized {key} 预览（前 500 字）---")
            print(text[:500] + ("..." if len(text) > 500 else ""))
    print("=" * 60)
    return {"report": report, "optimized": opt}


step3 = print_optimization_summary(REPORT_FILE, OPTIMIZED_FILE)

Planner Prompt 优化报告摘要
backend: textgrad_graph
executor_model: qwen-plus
optimizer_model: qwen-plus

--- decomposition (L1 任务拆解) ---
  optimizer: textgrad_graph
  baseline_dev: 0.95  （优化前 dev 均分）
  best_dev:     0.95  （最终保留的 dev 均分）
  step1 [拒绝] train=0.950 candidate_dev=0.950 failures=0
    （无失败 case，本步未更新 prompt）

--- routing (L2 子任务路由) ---
  optimizer: textgrad_graph
  baseline_dev: 1.0  （优化前 dev 均分）
  best_dev:     1.0  （最终保留的 dev 均分）
  step1 [拒绝] train=0.950 candidate_dev=1.000 failures=0
    （无失败 case，本步未更新 prompt）

--- optimized decomposition_prompt 预览（前 500 字）---
##角色：你是一个高度熟练的任务拆解专家，专注于将用户输入拆解为粒度适当的子任务。
##任务：根据用户的输入，分析目标以及智能体团队中所有智能体的能力，通过将用户输入分解为智能体团体能力能够覆盖的信息明确的子任务，以帮助有效地实现用户的目标。
##背景信息（关于具体任务拆解所需要知道的相关信息及要求）：
    {background_info}
##智能体团队（可用于完成子任务的所有智能体能力）：
    {agent_team}
##步骤：
    1. **理解目标**：根据所有用户输入确定用户想要实现的核心目标和结果。
    2. **考虑背景信息**：提供的背景信息中包含拆解任务所需要的额外的领域知识，可用作参考。
    3. **定义子任务**：根据目标的复杂程度将任务拆解为子任务。
      3.1 **任务拆解原则**：
        - 简单：每个子任务都是一个能够匹配智能体团队能力的单一且信息明确的祈使句指

## Step 4　优化后复评（对比提升）

加载 `optimized/zh.json`，再跑 L1/L2，与 Step1 对比。

In [11]:
step4 = await run_baseline_l1_l2(split=DEV_SPLIT, use_optimized=True)
print_l1_l2_report("Step4 优化后", step4)

print("\n=== 对比（dev）===")
print(f"  L1 decomposition: {step1['l1'].average_score:.3f} → {step4['l1'].average_score:.3f}")
print(f"  L2 routing:       {step1['l2'].average_score:.3f} → {step4['l2'].average_score:.3f}")

after_file = OUTPUT_DIR / "after_l1_l2_dev.json"
after_file.write_text(
    json.dumps({"l1": step4["l1"].to_dict(), "l2": step4["l2"].to_dict()}, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print(f"\n已保存优化后报告: {after_file}")


=== Step4 优化后 ===
L1 decomposition: avg=0.950 cases=2
  [PASS] case-08 score=0.900 subtasks=3
  [PASS] case-09 score=1.000 subtasks=2
L2 routing:       avg=1.000 cases=2
  [PASS] case-08 score=1.000 T1->WeatherAgent, T2->HotelAgent, T3->RestaurantAgent
  [PASS] case-09 score=1.000 T1->FlightAgent, T2->HotelAgent

=== 对比（dev）===
  L1 decomposition: 0.950 → 0.950
  L2 routing:       1.000 → 1.000

已保存优化后报告: D:\myproject\mira-ai-lab\agent-systems-book\Chapter-11\data\benchmark\travel_planner\after_l1_l2_dev.json


## 附录

### CLI 等价命令

```powershell
# Step 1 / Step 4：评测
python scripts/eval_travel_decomposition.py --split dev --no-use-optimized
python scripts/eval_travel_routing.py --split dev --no-use-optimized

# Step 2：与 Notebook 默认 smoke 一致
python scripts/optimize_travel_planner.py --backend textgrad_graph --objective l1_l2 --max-steps 1 --train-split dev --dev-split dev

# 正式优化（推荐学习完成后尝试）
python scripts/optimize_travel_planner.py --backend textgrad_graph --objective l1_l2 --max-steps 3 --train-split train --dev-split dev
```

### 本教程 vs 其它优化模式

| 模式 | 命令特征 | 适合场景 |
|------|----------|----------|
| **本教程（Planner B1）** | `--backend textgrad_graph --objective l1_l2` | 先打稳拆解与路由，成本低 |
| local | `--backend local` | 不用 textgrad 包，自写闭环 |
| textgrad_lib | `--backend textgrad_lib` | 失败文本 + TGD，非计算图 |
| **Planner B2 / E2E** | `--objective e2e` | 用完整编排分数优化，慢、贵 |

### 产物路径

| 文件 | 内容 |
|------|------|
| `data/benchmark/travel_planner/optimized/zh.json` | 优化后的 `decomposition_prompt`、`agent_routing` |
| `data/benchmark/travel_planner/planner_textgrad_graph_optimization_report.json` | 逐步 ACCEPT/REJECT 与得分 |
| `baseline_l1_l2_dev.json` / `after_l1_l2_dev.json` | Notebook Step 1 / 4 保存的评测结果 |